# 05 · Recorridos narrativos

Define recorridos manualmente (sucesión de paradas con coordenadas) y produce un mapa estilizado por recorrido + uno maestro de emociones.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'src'))

import corpus, spatial_extraction, affect_loader, cross_analysis, viz_maps

cfg = corpus.load_config('configs/sin_remedio.yaml')
parrafos = corpus.parse_from_config('data/corpus.txt', cfg)

with open(cfg['lugares_catalogo_path']) as f: cat_lug = json.load(f)
df_esp = spatial_extraction.extract(parrafos, cat_lug)
df_emo = affect_loader.load('data/extraccion_emocional.xlsx')
cruce = cross_analysis.cross_spatial_affect(df_esp, df_emo)

polaridad_fn = cross_analysis.make_polarity_fn(cfg['polaridad'])
df_largo = cross_analysis.long_format(cruce, polaridad_fn)
pivot = cross_analysis.pivot_place_category(df_largo,
    cfg['polaridad']['Negativa'], cfg['polaridad']['Positiva'], cfg['polaridad']['Tensión social'])

emo_dom = cross_analysis.dominant_emotion_by_place(pivot, cfg['afectivas_dominantes'])
print(f'Emoción dominante calculada para {len(emo_dom)} lugares')

## 1. Definir recorridos

Cada recorrido es una secuencia de paradas. Para Sin remedio, hay 9 grandes desplazamientos del protagonista.

In [ ]:
# Estructura: cada parada = (nombre_mostrar, lat, lon, lugar_canonico_para_emo)
RECORRIDOS = [
    {
        'id': 1, 'titulo': 'La noche del crimen',
        'subtitulo': 'Paseo nocturno por la Carrera Trece',
        'capitulo': 'CAPÍTULO I', 'medio': 'A pie',
        'paradas': [
            ('Apartamento (La Soledad)', 4.6300, -74.0750, 'Apartamento (de Escobar / Fina)'),
            ('Carrera Trece', 4.6244, -74.0680, 'Carrera Trece'),
            ('Bar El Oasis', 4.6360, -74.0680, 'Bar El Oasis'),
            ('Apartamento (regreso)', 4.6300, -74.0750, 'Apartamento (de Escobar / Fina)'),
        ],
    },
    # … resto de recorridos …
]

# Para el proyecto Sin remedio, cargar los 9 recorridos completos desde JSON:
recorridos_path = Path('data/recorridos.json')
if recorridos_path.exists():
    RECORRIDOS = json.loads(recorridos_path.read_text(encoding='utf-8'))

print(f'Recorridos definidos: {len(RECORRIDOS)}')

## 2. Generar mapa por recorrido

In [ ]:
emo_color = {k: '#'+v for k,v in cfg['emo_color'].items()}

for rec in RECORRIDOS:
    paradas_viz = []
    for nombre, lat, lon, canon in rec['paradas']:
        emo = emo_dom.get(canon, {}).get('dominante') if canon else None
        paradas_viz.append({'nombre': nombre, 'lat': lat, 'lon': lon, 'emocion': emo})

    out = f"outputs/recorrido_{rec['id']:02d}.png"
    viz_maps.route_map(paradas_viz, out, cfg['geometria_base'],
        title=f"Recorrido {rec['id']} · {rec['titulo']}",
        subtitle=f"{rec['capitulo']} · {rec['medio']}",
        emo_color=emo_color,
    )
    print(f'  ✓ {out}')

## 3. Mapa maestro de emociones

In [ ]:
# Lista de todos los lugares con su emoción dominante
coords = df_esp.drop_duplicates('Lugar nombrado (canónico)').set_index(
    'Lugar nombrado (canónico)')[['Latitud (aprox.)','Longitud (aprox.)']].to_dict('index')

places_emo = []
for lugar, info in emo_dom.items():
    c = coords.get(lugar, {})
    if not c.get('Latitud (aprox.)'): continue
    places_emo.append({
        'name': lugar,
        'lat': c['Latitud (aprox.)'], 'lon': c['Longitud (aprox.)'],
        'idx_desencanto': info['idx_desencanto'],
        'total_menciones': info['total'],
        'dominante': info['dominante'],
    })

viz_maps.heat_map(places_emo, 'outputs/mapa_maestro_emociones.png',
                  geom=cfg['geometria_base'],
                  title='Mapa maestro de emociones · ' + cfg['proyecto']['nombre'])